# 1. Imports and other

Ruta raíz del proyecto

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
print(PROJECT_ROOT)

c:\Users\Carlos\Documents\Ingenieria_informatica\Proyectos_datascience\3_prediccion_precios_casas


Añadir raíz al sys

In [2]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

Imports

In [3]:
import pandas as pd
from src.utils import simple_report

# 2. Read clean data

In [4]:
df = pd.read_csv(PROJECT_ROOT / 'data' / 'clean' / 'clean_train.csv')

In [5]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NoAlleyAccess,Reg,Lvl,AllPub,...,0,NoPool,NoFence,NoMisceFeat,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NoAlleyAccess,Reg,Lvl,AllPub,...,0,NoPool,NoFence,NoMisceFeat,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NoAlleyAccess,IR1,Lvl,AllPub,...,0,NoPool,NoFence,NoMisceFeat,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NoAlleyAccess,IR1,Lvl,AllPub,...,0,NoPool,NoFence,NoMisceFeat,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NoAlleyAccess,IR1,Lvl,AllPub,...,0,NoPool,NoFence,NoMisceFeat,0,12,2008,WD,Normal,250000


# 3. Preprocess

## 3.1 Explore

Primero decidiremos que hacer con los features categóricos

Esta celda sirve para ir viendo los features que tenemos

In [6]:
df_categorical = df.select_dtypes(include='str')
df_categorical.iloc[:5, 40:]

,MiscFeature,SaleType,SaleCondition
0,NoMisceFeat,WD,Normal
1,NoMisceFeat,WD,Normal
2,NoMisceFeat,WD,Normal
3,NoMisceFeat,WD,Abnorml
4,NoMisceFeat,WD,Normal


Celda para revisar categorías de cada feature

In [7]:
simple_report(df, 'PoolQC', show_categories=True)


Nulls: 0
Dtype: str
Total categories: 4
Categories: ['NoPool', 'Ex', 'Fa', 'Gd']



## 3.2 Decision

Features a transformar:

- LotShape
- ExterQual
- ExterCond
- BsmtQual
- BsmtCond
- HeatingQC
- KitchenQual
- FireplaceQu
- GarageQual
- GarageCond
- PoolQC

Casi todos los features anteriores tienen las mismas categorías jerarquicas, solo LotShape tiene diferentes

In [8]:
HIERARCHIES_DICT = {
    'Po': 1,
    'Fa': 2,
    'TA': 3,
    'Gd': 4,
    'Ex': 5
}

Diccionario con categorías casi identicas (solo diferencian en el nombre de cuando no contienen el feature)

In [9]:
list_hierarchies_same_features = [
    'ExterQual',
    'ExterCond',
    'BsmtQual',
    'BsmtCond',
    'HeatingQC',
    'KitchenQual',
    'FireplaceQu',
    'GarageQual',
    'GarageCond',
    'PoolQC'
]

# 4. LotShape

Ver forma de categorías

In [10]:
df['LotShape'].unique()

<StringArray>
['Reg', 'IR1', 'IR2', 'IR3']
Length: 4, dtype: str

## 4.1 Replace

In [11]:
dict_to_map = {
    'IR3': 1,
    'IR2': 2,
    'IR1': 3,
    'Reg': 4
}

In [12]:
df['LotShape'] = df['LotShape'].map(dict_to_map)

In [13]:
df['LotShape'].head()

0    4
1    4
2    3
3    3
4    3
Name: LotShape, dtype: int64

# 5. Other hierarchies categorical features

Recorrer todos los features con la misma jerarquía de categorías. Hacer mismo reeemplazo, cuando se encuentre con la categoría de no tener el feature, quedará NaN, asi que lo cambiamos por un 0

In [14]:
for feat in list_hierarchies_same_features:
    df[feat] = df[feat].map(HIERARCHIES_DICT).fillna(0)

Volver a revisar los features categóricos

In [15]:
df_categorical = df.select_dtypes(include='str')
df_categorical.iloc[:5, 15:]

,Exterior2nd,MasVnrType,Foundation,BsmtExposure,BsmtFinType1,BsmtFinType2,Heating,CentralAir,Electrical,Functional,GarageType,GarageFinish,PavedDrive,Fence,MiscFeature,SaleType,SaleCondition
0,VinylSd,BrkFace,PConc,No,GLQ,Unf,GasA,Y,SBrkr,Typ,Attchd,RFn,Y,NoFence,NoMisceFeat,WD,Normal
1,MetalSd,NoMasVnr,CBlock,Gd,ALQ,Unf,GasA,Y,SBrkr,Typ,Attchd,RFn,Y,NoFence,NoMisceFeat,WD,Normal
2,VinylSd,BrkFace,PConc,Mn,GLQ,Unf,GasA,Y,SBrkr,Typ,Attchd,RFn,Y,NoFence,NoMisceFeat,WD,Normal
3,Wd Shng,NoMasVnr,BrkTil,No,ALQ,Unf,GasA,Y,SBrkr,Typ,Detchd,Unf,Y,NoFence,NoMisceFeat,WD,Abnorml
4,VinylSd,BrkFace,PConc,Av,GLQ,Unf,GasA,Y,SBrkr,Typ,Attchd,RFn,Y,NoFence,NoMisceFeat,WD,Normal


Tales features quedarán para el one-hot-encoding del modelado, así que guardamos la información como está

# 6. Delete Id

Los Id's no nos sirven de mucho, asi que los eliminamos

In [16]:
df.drop(columns=['Id'], inplace=True)

# 7. Save data

In [17]:
df.to_csv(PROJECT_ROOT / 'data' / 'processed' / 'processed_train.csv', index=False)